# Vanda Orchid Disease Classifier — Colab training

**R26-SE-018 · Component 1**

Run the cells in order. Total time: roughly **50–70 minutes**, most of it in step 6.

| Step | What it does | Time |
|---|---|---|
| 1 | Confirm a GPU is attached | 10 s |
| 2 | Mount Google Drive | 30 s |
| 3 | Unpack `orchid_colab.zip` | 1 min |
| 4 | Split 667 photos into 533 / 67 / 67 | 1 min |
| 5 | Augment the training set 54× | 6–10 min |
| 6 | **Train the disease classifier** | 30–50 min |
| 7 | Evaluate on the test set | 1 min |
| 8 | Download the model and results | 1 min |

---

## Before you start

**1. Turn on the GPU.** Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save.
Without this, training takes hours instead of minutes.

**2. Upload the data.** In your browser, go to [drive.google.com](https://drive.google.com) and drag
`orchid_colab.zip` (134 MB) into **My Drive**. Wait for it to finish uploading before running step 3.

Uploading through Drive rather than through Colab's file picker matters: a Drive upload resumes if
your connection drops, and it survives a Colab session restart, so you never upload twice.

## Step 1 — confirm the GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPU       :', [g.name for g in gpus] or 'NONE')

if not gpus:
    print('\n*** NO GPU ***')
    print('Runtime -> Change runtime type -> T4 GPU -> Save, then re-run this cell.')
    print('Training on CPU here would take several hours.')
else:
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

print('\nNOTE: check this TensorFlow version against your local one (2.21.0).')
print('A .keras file saved by a newer TF may not load on an older one.')
print('train.py also saves .weights.h5 as a fallback if that happens.')

## Step 2 — mount Google Drive

A popup will ask you to authorise. Choose your account and allow access.

Drive is used for two things: reading the uploaded zip, and **checkpointing the model as it trains**.
If the Colab session drops mid-run, checkpointing to Drive means you lose one epoch instead of the
whole run.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

ZIP = Path('/content/drive/MyDrive/orchid_colab.zip')
OUT = Path('/content/drive/MyDrive/orchid_models')   # models land here
OUT.mkdir(parents=True, exist_ok=True)

if ZIP.exists():
    print('found:', ZIP, '({:.0f} MB)'.format(ZIP.stat().st_size / 1e6))
else:
    print('*** NOT FOUND:', ZIP)
    print('Upload orchid_colab.zip to the TOP LEVEL of My Drive (not in a folder),')
    print('wait for the upload to complete, then re-run this cell.')
print('models will be saved to:', OUT)

## Step 3 — unpack

Everything is unzipped to `/content/`, **not** to the mounted Drive folder. Per-file latency on the
Drive mount would starve the GPU and turn a 90-second epoch into 15 minutes. Only the finished model
goes back to Drive.

In [ ]:
import shutil, zipfile, time
from pathlib import Path

t0 = time.time()
for p in ('/content/pkg', '/content/data', '/content/scripts', '/content/models'):
    shutil.rmtree(p, ignore_errors=True)

with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/pkg')

# Lay the files out so the scripts' own relative paths resolve:
#   /content/scripts/train.py  ->  COMPONENT_ROOT = /content
#   /content/data/processed/   ->  where it looks for the images
Path('/content/data').mkdir(exist_ok=True)
shutil.move('/content/pkg/processed', '/content/data/processed')
shutil.move('/content/pkg/scripts', '/content/scripts')
shutil.move('/content/pkg/data_extra/severity_labels.csv',
            '/content/data/severity_labels.csv')
Path('/content/models').mkdir(exist_ok=True)

print('unpacked in {:.0f}s\n'.format(time.time() - t0))
total = 0
for d in sorted(Path('/content/data/processed').iterdir()):
    n = len(list(d.glob('*.jpg')))
    total += n
    print('  {:<26} {:>4} images'.format(d.name, n))
print('  {:<26} {:>4}'.format('TOTAL', total))
assert total == 667, 'Expected 667 images, found {} -- re-upload the zip'.format(total)
print('\nOK')

## Step 4 — split 80 / 10 / 10

Stratified and seeded with 42, so this produces **exactly the same split as on your laptop**. That is
what makes the counts and the leakage evidence in your report still true for the Colab-trained model.

In [ ]:
!cd /content/scripts && python split_dataset.py

## Step 5 — augment the TRAINING set only

533 originals × 54 = 28,782 files. Validation and test are copied across **unmodified**, because they
exist to answer "how does this do on a photo a grower actually takes?"

Takes 6–10 minutes. This is the slowest step before training.

In [ ]:
%%time
import subprocess, shutil
from pathlib import Path

for cls in ['black_leaf_spot', 'phyllosticta_leaf_spot', 'healthy']:
    print('\n=== augmenting', cls, '===')
    subprocess.run([
        'python', 'augment_dataset.py', 'augment',
        '--input',  '/content/data/split/train/' + cls,
        '--output', '/content/data/split_augmented/train/' + cls,
        '--disease', cls,
        '--labels', '/content/data/severity_labels.csv',
    ], cwd='/content/scripts', check=True)

# validation and test go across untouched
for split in ['validation', 'test']:
    dst = Path('/content/data/split_augmented') / split
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree('/content/data/split/' + split, dst)

print('\n=== final counts ===')
for split in ['train', 'validation', 'test']:
    for d in sorted((Path('/content/data/split_augmented') / split).iterdir()):
        if d.is_dir():
            print('  {:<12} {:<26} {:>6}'.format(split, d.name, len(list(d.glob('*.jpg')))))

### Leakage check — run this and screenshot the output

This is the evidence for your results chapter. It proves no photograph contributed to both training
and evaluation.

In [ ]:
!cd /content/scripts && python check_leakage.py

## Step 6 — train

**Run the smoke test first.** One epoch on three batches, about 60 seconds. It proves the whole path
works before you commit 40 minutes of GPU time to it.

In [ ]:
# SMOKE TEST -- ~60 seconds. The accuracy printed here is meaningless by design.
!cd /content/scripts && python train.py --smoke-test --out /content/models

If that finished without an error, run the real thing.

Checkpoints go to **Drive**, so a dropped session costs one epoch rather than the whole run.
EarlyStopping will usually stop before the full 25 + 12 epochs.

**Keep this browser tab visible.** Colab disconnects idle sessions.

In [ ]:
%%time
!cd /content/scripts && python train.py \
    --data /content/data/split_augmented \
    --out  /content/drive/MyDrive/orchid_models

## Step 7 — evaluate on the test set

**These are your reportable numbers.** Per-class precision, recall and F1 on 67 real, unmodified
photographs the model has never seen.

Quote **macro F1** as the headline, not overall accuracy — with only 15 Black Leaf Spot test images,
one error moves that class's recall by 6.7 points while barely touching overall accuracy.

In [ ]:
!cd /content/scripts && python evaluate.py \
    --data /content/data/split_augmented \
    --models /content/drive/MyDrive/orchid_models

### Choose the unknown-disease threshold

Run the same evaluation on **validation** and pick your operating threshold from that table.
Choosing it from the test table would be tuning on the test set, and your reported figures would be
optimistic.

In [ ]:
!cd /content/scripts && python evaluate.py --split validation \
    --data /content/data/split_augmented \
    --models /content/drive/MyDrive/orchid_models

In [ ]:
# Show the confusion matrix inline, so you can screenshot it for the report.
from IPython.display import Image, display
from pathlib import Path

fig = Path('/content/drive/MyDrive/orchid_models/confusion_matrix_test.png')
display(Image(str(fig))) if fig.exists() else print('not found -- run step 7 first')

## Step 8 — download

Everything is already in `MyDrive/orchid_models`. This cell also downloads it straight to your
laptop. Put the files in `ml-models/disease_detection/models/`.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/orchid_results', 'zip',
                    '/content/drive/MyDrive/orchid_models')
print('Contains: disease_model.keras, class_names.json, metrics_test.json,')
print('          evaluation_report_test.txt, confusion_matrix_test.png\n')
files.download('/content/orchid_results.zip')

---

## Optional — architecture comparison

Answers "why did you choose this model?" with measurements instead of reputation. Trains five
architectures under identical conditions and reports macro-F1 alongside model size and inference
latency.

**Only run this if you have spare time** — it takes about 40 minutes. Your priority is a trained
disease model and a trained severity model. If you are short on time, skip it and justify
MobileNetV2 on the deployment argument alone: the target is a grower's phone in a shade house,
possibly offline, so size and latency are hard constraints.

It scores on **validation**, never test.

In [ ]:
# Reduced comparison: the two realistic candidates plus the no-pretraining control.
!cd /content/scripts && python compare_models.py \
    --models mobilenetv2 efficientnetb0 scratch_cnn \
    --epochs 5 \
    --data /content/data/split_augmented \
    --out  /content/drive/MyDrive/orchid_models

---

## Later — the severity model (Model 2)

Run this **after** you have filled in `data/severity_labels.csv`. Upload the finished CSV to
`MyDrive/severity_labels.csv`, then run the two cells below.

Severity trains on diseased images only — a healthy plant has no grade to predict.

In [ ]:
# Copy your finished labels in, then check how many are usable.
import shutil
shutil.copy('/content/drive/MyDrive/severity_labels.csv',
            '/content/data/severity_labels.csv')
!cd /content/scripts && python train_severity.py --check

In [ ]:
%%time
!cd /content/scripts && python train_severity.py \
    --data /content/data/split_augmented \
    --labels /content/data/severity_labels.csv \
    --out /content/drive/MyDrive/orchid_models

---

## If something goes wrong

| Symptom | Fix |
|---|---|
| `NOT FOUND: orchid_colab.zip` | The zip must be at the top level of My Drive, not inside a folder. Wait for the upload to finish. |
| No GPU listed in step 1 | Runtime → Change runtime type → T4 GPU → Save. Then re-run from step 1. |
| Session disconnected mid-training | Re-run steps 2–5, then step 6. Checkpoints in Drive mean you resume near where you stopped. |
| `assert total == 667` fails | The zip uploaded incompletely. Delete it from Drive and re-upload. |
| Step 5 very slow | You are writing to the Drive mount instead of `/content`. Re-run step 3. |
| Out of memory during training | Add `--batch-size 16` to the train command. |

**Colab free tier disconnects idle sessions.** Keep the tab open and visible while training.